# Cheatsheet Maker in Google Colab

Run the package in **your own runtime with your own Gemini key**. No web server or Google Drive mount is needed.

Choose **Runtime → Change runtime type → GPU** if you want GPU acceleration. The package also works with a CPU runtime. Availability and speed depend on your runtime and documents.

This notebook saves no outputs or credentials in the repository. Review notebook outputs before sharing it. Extraction runs in your runtime; generation sends extracted content and custom prompts to Gemini.


## 1. Clone and install

Replace the repository URL below with the URL of your clone/fork. Install from a release tag or commit when sharing a reproducible notebook. The first model-based parse may download Docling weights.


In [ ]:
from pathlib import Path
import subprocess

repo_url = "https://github.com/OWNER/REPOSITORY.git"
repo_directory = Path("/content/cheatsheet-maker")

if "OWNER/REPOSITORY" in repo_url:
    raise ValueError("Replace repo_url with the actual GitHub repository URL.")
if not repo_directory.exists():
    subprocess.run(["git", "clone", "--", repo_url, str(repo_directory)], check=True)


In [ ]:
%pip install -e /content/cheatsheet-maker


## 2. Upload and parse materials

Upload up to 10 PDF, DOCX, or PPTX files, with 25 MB per file and 100 MB total. All selected files become one cheatsheet. Parsing does not need a Gemini key.


In [ ]:
from google.colab import files
from cheatsheet_maker import CheatsheetMaker

uploaded = files.upload()
maker = CheatsheetMaker(device="auto", ocr=True, formula_enrichment=True)
markdown = maker.parse(uploaded)

output_directory = Path("/content/cheatsheet-output")
output_directory.mkdir(parents=True, exist_ok=True)
markdown_path = output_directory / "parsed_notes.md"
markdown_path.write_text(markdown, encoding="utf-8")
print(f"Parsed {len(uploaded)} file(s). Saved Markdown to {markdown_path}.")


## 3. Generate with your own key

Add `GEMINI_API_KEY` in Colab's **Secrets** panel (key icon) and allow this notebook to access it. Never paste the key into source cells or print it.

This example generates LaTeX without requiring Tectonic. For a PDF, install the Tectonic executable using its [official instructions](https://tectonic-typesetting.github.io/en-US/install.html), then set `compile_pdf=True`. The package does not install executables automatically.

Changing this cell's prompts or layout reuses the parsed Markdown instead of running Docling again. Gemini requests still consume your account's quota. Choose a model available to your account.


In [ ]:
from google.colab import userdata
from cheatsheet_maker import FormatSettings

with CheatsheetMaker(
    api_key=userdata.get("GEMINI_API_KEY"),
    model="gemini-3.6-flash",
) as generator:
    result = generator.generate_from_markdown(
        markdown,
        settings=FormatSettings(columns=3, margin="1cm"),
        output_dir=output_directory,
        extra_classifier="Focus on key definitions, theorems, and formulas.",
        extra_summarizer="Keep explanations concise.",
        compile_pdf=False,
    )

print(f"LaTeX: {result.tex_path}")
if result.pdf_path is not None:
    print(f"PDF: {result.pdf_path}")


## 4. Download your results

Runtime storage is temporary. Download your files before disconnecting. LaTeX output and parsed Markdown may contain private lecture content; keep them private as needed.


In [ ]:
files.download(str(result.tex_path))
if result.pdf_path is not None:
    files.download(str(result.pdf_path))

# Optional: download extracted text to reuse it in another session.
# files.download(str(markdown_path))
